# RAG vs Fine-Tuning Strategy Map

This notebook compares **RAG**, **fine-tuning-style behavior adaptation**, and **hybrid prompting** with real model calls when a provider key is configured. It does not run a real fine-tuning job; instead, it uses few-shot examples to approximate the kind of stable behavior that fine-tuning is often used to learn.


## Learning Objectives

By the end of this notebook, you should be able to:

- Explain what RAG, fine-tuning, and hybrid strategies change.
- Distinguish parametric from non-parametric knowledge.
- Build prompts that change one ingredient at a time.
- Diagnose missing facts, missing citations, and inconsistent behavior.


## Strategy Vocabulary

Think of the three strategies as answers to different change problems. **RAG** changes what the model can see during a request, so it is strongest when facts are fresh, citeable, or frequently updated. **Fine-tuning** changes model behavior, so it is strongest when the issue is tone, format, refusal style, or repeated task behavior. A **hybrid** system keeps retrieved evidence for factual grounding while using examples or adapted weights to make responses consistent.

A common mistake is to ask fine-tuning to memorize living policy. That creates maintenance risk: the model can sound confident while repeating stale information. Another common mistake is to ask RAG to fix style problems. Retrieval can provide facts, but it does not automatically make the answer concise, empathetic, or consistent.


## Definitions With A Concrete Support Example

**Parametric knowledge** is information stored inside model weights. It is useful for broad language patterns and common facts, but it is hard to audit and hard to update quickly. Example: a base model may know what an invoice is, but it should not be trusted to know the current refund policy for this course scenario.

**Non-parametric knowledge** is information supplied outside the model weights, usually through retrieval, tools, or prompt context. It is easier to update and cite. Example: the refund policy document can be edited today, embedded again, retrieved tomorrow, and cited in the answer.

**Behavior adaptation** means changing how the model responds rather than what facts it has access to. A model can learn to be concise, to refuse unsafe requests carefully, or to return JSON consistently. That is different from learning a specific policy. In the notebooks, few-shot examples stand in for this behavior adaptation so participants can compare the strategy before training anything.


## What Changes Across Strategies

Before writing prompts, separate the two places where an LLM system can get information: the model weights and the runtime context. RAG changes the runtime context; fine-tuning changes behavior stored in model parameters or adapter parameters; hybrid systems use both.

The figure below frames the three options as ways to take an exam. Fine-tuning is a closed-book exam: knowledge is baked into the model at training time. RAG is an open-book exam: the model reads external documents at answer time. RAFT, the method proposed in the paper, combines both by training the model to use retrieved documents well. The hybrid in this repository is lighter: it combines retrieved documents with style examples in the prompt instead of training.

<div align="center">
  <img src="assets/01-raft-closed-vs-open-book.png" alt="Closed-book, open-book, and RAFT approaches to answering a query" width="900" style="max-width: 100%; height: auto;" />
  <p><em>Closed book (knowledge in the weights), open book (documents at answer time), and RAFT (a model trained to use documents). Source: <a href="https://arxiv.org/abs/2403.10131" target="_blank" rel="noopener noreferrer">Zhang et al., 2024</a>, CC BY 4.0</em></p>
</div>


In [ ]:
from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    call_model,
    style_messages,
)

## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells can now focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

## Three Prompting Conditions

We will ask the same user question three ways:

- baseline: no retrieved evidence and no style examples;
- retrieval-aware: retrieved policy context is supplied;
- style-aware: response examples are supplied, but no fresh policy context is supplied.

The contrast shows why RAG and fine-tuning solve different problems.


## How To Read The Comparison

The three prompt variants are not meant to compete as polished production prompts. They are controlled experiments. The baseline tells us what the model does with only a general instruction. The retrieval-aware version tells us how much better the answer becomes when policy evidence is added. The style-aware version tells us whether examples can steer tone and structure without adding fresh facts.

A practical rule of thumb: if the model knows the tone but misses the policy, add retrieval. If it cites correctly but sounds inconsistent, improve examples or prompts. If both evidence and style matter, test a hybrid path before investing in a new fine-tuning run.


The next comparison intentionally changes one ingredient at a time. The baseline has only the user question, the retrieval-aware prompt adds policy evidence, and the style-aware prompt adds examples without policy context. This makes the failure modes easier to name: missing facts, missing citations, or inconsistent behavior.


Few-shot prompting turns approved examples into alternating user and assistant messages. The examples are not new model weights; they are context supplied at request time.


## Build Few-Shot Style Messages

Few-shot messages make behavior examples visible in the prompt. They do not update model weights.


Start with one request so the prompt differences stay easy to compare.


In [ ]:
question = REQUESTS[0]["question"]
retrieved = [DOCUMENTS[0]]
style_seed = STYLE_EXAMPLES[:2]

The baseline has no domain evidence and no style examples.


In [ ]:
baseline_messages = [
    {"role": "system", "content": "You are a concise support assistant."},
    {"role": "user", "content": question},
]

The retrieval-aware version adds one citeable policy document to the user message.


In [ ]:
retrieval_messages = [
    {
        "role": "system",
        "content": "Answer only from the provided policy context. Cite source IDs in square brackets.",
    },
    {
        "role": "user",
        "content": f"Policy context:\n[refund-policy] {retrieved[0]['text']}\n\nQuestion: {question}",
    },
]

The style-aware version adds examples but no fresh policy context, which is why it can imitate tone but may miss facts.


In [ ]:
style_messages_only = [
    {
        "role": "system",
        "content": "Match the style of the examples. Do not invent policy details.",
    },
    *style_messages(style_seed),
    {"role": "user", "content": question},
]

Collect the variants in one dictionary so the same model-call loop can compare them.


In [ ]:
prompt_variants = {
    "baseline": baseline_messages,
    "retrieval_aware": retrieval_messages,
    "style_aware": style_messages_only,
}

for name, messages in prompt_variants.items():
    print(name, "messages=", len(messages))

## Call The Model

In dry mode, the function returns deterministic teaching outputs. With `GROQ_API_KEY` or `MISTRAL_API_KEY`, the same cell calls the live model.


In [ ]:
dry_outputs = {
    "baseline": "A refund may be possible, but I need the invoice and payment details.",
    "retrieval_aware": "Duplicate payments can be refunded when both payments belong to the same invoice. Provide the invoice number and both payment references within 30 days. [refund-policy]",
    "style_aware": "Ask for the invoice number and both payment references, then explain the next step in a concise support tone.",
}

responses = {}
for name, messages in prompt_variants.items():
    responses[name] = call_model(messages, dry_response=dry_outputs[name])
    print()
    print(name.upper(), responses[name].source)
    print(responses[name].text)

## Interpret The Difference

The retrieval-aware answer can cite a specific source because the prompt supplied source text. The style-aware answer can mimic response behavior, but it still cannot prove the current refund rule unless policy evidence is present.


In [ ]:
mfa_question = REQUESTS[2]["question"]
mfa_context = next(
    document for document in DOCUMENTS if document["doc_id"] == "mfa-reset"
)
messages = [
    {
        "role": "system",
        "content": "Answer only from policy context and cite source IDs.",
    },
    {
        "role": "user",
        "content": f"Policy context:\n[mfa-reset] {mfa_context['text']}\n\nQuestion: {mfa_question}",
    },
]
reply = call_model(
    messages,
    dry_response="MFA can be reset only after identity verification by support. Do not send identity documents or backup codes through chat. [mfa-reset]",
)
print(reply.source, reply.text)

## Check Your Understanding

1. A policy changes every week, but the response style is already acceptable. Which strategy should be tried first?
2. In a RAG-only system, which part changes at request time: the model weights or the retrieved context?
3. Which prompt variant (baseline, retrieval-aware, or style-aware) mainly tests behavior adaptation?
4. Which response should be easiest to audit?

<details>
<summary>Show Reference Answers</summary>

1. Start with RAG. The main problem is freshness and citeable policy evidence, not learned response behavior.
2. The retrieved context changes at request time. The model weights stay the same in a RAG-only system.
3. The style-aware variant. It adds response examples but no policy context, so any change in the answer comes from behavior conditioning rather than new facts.
4. The retrieval-aware response is easiest to audit because it asks the model to cite source IDs from the provided policy context.

</details>


## Further Reading

- [Fine-Tuning or Retrieval? Comparing Knowledge Injection in LLMs](https://arxiv.org/abs/2312.05934) - Ovadia et al., 2023
